# 🎓 Pedagogical Knowledge Graph Extractor — GitHub + Colab

**One-click deployment from GitHub to Google Colab with full Streamlit UI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navadeepkiran/Pedagogical_flow_extractor/blob/main/PedagogicalKG_GitHub_Colab.ipynb)

## 🚀 Features
- ✅ **Direct GitHub Integration** — Clone and run in seconds
- ✅ **Full Streamlit Web UI** — Same interface as localhost
- ✅ **Environment Variables** — Secure API key handling
- ✅ **Multi-language Support** — English, Hinglish, Tenglish
- ✅ **Interactive Graphs** — PyVis visualization with communities
- ✅ **Public Access** — Share your analysis via ngrok URL

## 🎯 Quick Start
1. **Set your API key** below (Cell 3)
2. **Run all cells** (Runtime → Run all)
3. **Click the ngrok URL** when ready
4. **Upload videos** and explore knowledge graphs!

---

## Step 1: Install Dependencies

In [ ]:
%%capture
print("📦 Installing dependencies...")

# Core ML libraries
!pip install -q openai-whisper torch torchvision torchaudio
!pip install -q ffmpeg-python pydub yt-dlp
!pip install -q rake-nltk nltk networkx pyvis plotly
!pip install -q sentence-transformers groq pyyaml

# Web app & tunneling
!pip install -q streamlit pyngrok python-dotenv

# System dependencies  
!apt-get -qq update && apt-get -qq install -y ffmpeg

# NLTK data
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print('✅ Dependencies installed!')

## Step 2: Clone Repository from GitHub

In [ ]:
import os
import sys

# GitHub repository URL
GITHUB_REPO = "https://github.com/navadeepkiran/Pedagogical_flow_extractor.git"
PROJECT_DIR = "/content/PedagogicalFlowExtractor"

print(f"📥 Cloning repository: {GITHUB_REPO}")

# Remove existing directory if present
if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}

# Clone repository
!git clone {GITHUB_REPO} {PROJECT_DIR}

# Change to project directory
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f"\n✅ Repository cloned to: {PROJECT_DIR}")
print("\n📁 Project structure:")
!ls -la

## Step 3: Configure Environment Variables

**⚠️ REQUIRED: Set your Groq API key**

Get a free API key at: https://console.groq.com/keys

In [ ]:
import os

# ═══════════════════════════════════════════════════════════════
#  🔑 SET YOUR API KEY HERE
# ═══════════════════════════════════════════════════════════════

GROQ_API_KEY = "your_groq_api_key_here"  # ⚠️ UPDATE THIS!

# Optional: For tunnels longer than 2 hours
NGROK_AUTH_TOKEN = ""  # Get from https://dashboard.ngrok.com/

# ═══════════════════════════════════════════════════════════════

# Validate API key
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    print("❌ Please set your GROQ_API_KEY above!")
    print("   Get a free key at: https://console.groq.com/keys")
    exit(1)

# Set environment variables
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
if NGROK_AUTH_TOKEN:
    os.environ['NGROK_AUTH_TOKEN'] = NGROK_AUTH_TOKEN

# Create .env file
env_content = f"""GROQ_API_KEY={GROQ_API_KEY}
WHISPER_MODEL=small
WHISPER_LANGUAGE=null
LLM_MODEL=llama-3.3-70b-versatile
MAX_TOKENS=8192
STREAMLIT_PORT=8501
STREAMLIT_HOST=0.0.0.0
STREAMLIT_ENABLE_CORS=false
STREAMLIT_ENABLE_XSRF=false
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ Environment configured successfully!")
print(f"   API Key: {GROQ_API_KEY[:20]}...")
print(f"   Working Directory: {os.getcwd()}")

## Step 4: Validate Setup

In [ ]:
# Test imports and configuration
try:
    from utils.config import load_config, validate_config, get_api_key
    
    print("🔧 Validating configuration...")
    
    # Load config
    config = load_config()
    print(f"✅ Config loaded: {len(config)} sections")
    
    # Test API key
    api_key = get_api_key('groq')
    print(f"✅ API key loaded: {api_key[:15]}...")
    
    # Validate all components
    if validate_config():
        print("✅ Configuration validation passed!")
    else:
        print("❌ Configuration validation failed")
    
    # Test core imports
    from pipeline.normalizer import CodeMixedNormalizer
    from pipeline.graph_builder import GraphBuilder
    from app.streamlit_app import init_session_state
    
    print("✅ All modules imported successfully")
    
except Exception as e:
    print(f"❌ Setup validation failed: {e}")
    import traceback
    traceback.print_exc()

## Step 5: Setup ngrok Tunnel

In [ ]:
from pyngrok import ngrok, conf
import getpass

# Configure ngrok
if 'NGROK_AUTH_TOKEN' in os.environ and os.environ['NGROK_AUTH_TOKEN']:
    conf.get_default().auth_token = os.environ['NGROK_AUTH_TOKEN']
    print("✅ ngrok auth token configured")
else:
    print("ℹ️  No ngrok auth token — tunnel will expire in 2 hours")
    print("   For longer sessions, get a token at: https://dashboard.ngrok.com/")

# Kill existing tunnels
ngrok.kill()
print("✅ ngrok ready")

## Step 6: Launch Streamlit App

**🚀 This creates the web interface and public URL**

In [ ]:
import subprocess
import time
import threading
from IPython.display import display, HTML

# Kill any existing Streamlit processes
!pkill -f streamlit || true
time.sleep(3)

print("🚀 Starting Streamlit...")

# Start Streamlit in background
def run_streamlit():
    cmd = [
        "streamlit", "run", "app/streamlit_app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0", 
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ]
    subprocess.run(cmd, cwd=PROJECT_DIR)

# Start Streamlit in a separate thread
streamlit_thread = threading.Thread(target=run_streamlit, daemon=True)
streamlit_thread.start()

# Wait for Streamlit to start
print("⏳ Waiting for Streamlit to start...")
time.sleep(10)

# Create ngrok tunnel
try:
    public_url = ngrok.connect(8501, bind_tls=True)
    
    print("\n" + "="*80)
    print("🎉 SUCCESS! Your app is ready!")
    print("="*80)
    print(f"\n🌐 PUBLIC URL: {public_url}")
    print(f"\n📱 Click the link above to access your Pedagogical Knowledge Graph Extractor!")
    print("\n💡 Features available:")
    print("   • Upload video/audio files")
    print("   • Paste YouTube URLs")
    print("   • Interactive graph visualization")
    print("   • Multi-language support (English/Hindi/Telugu)")
    print("   • Export JSON and HTML reports")
    print("\n⏰ The app will stay running until you stop this Colab session")
    print("\n" + "="*80)
    
    # Create clickable link
    display(HTML(f'<h2><a href="{public_url}" target="_blank">🚀 Open Pedagogical Knowledge Graph Extractor</a></h2>'))
    
except Exception as e:
    print(f"\n❌ Error creating tunnel: {e}")
    print("\n🔄 Please try running this cell again")

## Step 7: Monitor & Control

Use these cells to check status, view logs, or restart if needed.

In [ ]:
# Check Streamlit status
print("🔍 Checking Streamlit processes:")
!ps aux | grep streamlit | grep -v grep || echo "No Streamlit processes found"

print("\n🌐 Active ngrok tunnels:")
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
    print(f"   {tunnel.public_url} -> {tunnel.config['addr']}")

In [ ]:
# View recent Streamlit logs
print("📋 Recent Streamlit logs:")
!tail -n 20 ~/.streamlit/logs/*.log 2>/dev/null || echo "No log files found yet"

In [ ]:
# Restart Streamlit (run this if the app stops working)
print("🔄 Restarting Streamlit...")

# Stop existing processes
!pkill -f streamlit || true
time.sleep(3)

# Start new process
streamlit_thread = threading.Thread(target=run_streamlit, daemon=True)
streamlit_thread.start()
time.sleep(5)

print("✅ Streamlit restarted")
print(f"🌐 Your app is still available at: {public_url}")

In [ ]:
# Stop everything (run this when you're done)
print("🛑 Stopping all services...")

!pkill -f streamlit || true
ngrok.kill()

print("✅ All services stopped")

---

## 📚 How to Use the App

Once you click the ngrok URL above, you'll see the Streamlit interface:

### 1. Input Options
- **📹 Upload File**: Click "Browse files" to upload .mp4, .mp3, .wav files
- **🔗 YouTube URL**: Paste any YouTube video link
- **📝 Direct Text**: Paste transcript text directly

### 2. Extraction Modes
- **Rule-Based**: Fast, good for CS/programming topics (~0.1s)
- **LLM-Powered**: Accurate for any subject using Groq API (~5s)

### 3. Interactive Features
- **🎨 Graph Visualization**: Zoom, pan, drag nodes
- **🔍 Search**: Find specific concepts
- **📊 Analytics**: PageRank scores, centrality metrics
- **🎯 Communities**: Color-coded concept clusters
- **📈 Timeline**: When concepts appear in the lecture
- **📥 Export**: Download JSON data and HTML visualizations

### 4. Multi-Language Support
The system automatically handles:
- **English**: Pure English lectures
- **Hinglish**: Hindi-English code-mixed ("Aaj hum arrays ke baare mein padhenge")
- **Tenglish**: Telugu-English code-mixed ("Binary tree oka data structure")

---

## 🛠️ Troubleshooting

### App not loading?
- Wait 30 seconds after starting
- Check "Monitor & Control" section above
- Restart with the restart cell

### "API key not set" error?
- Update `GROQ_API_KEY` in Step 3
- Re-run Step 3 and Step 6

### Tunnel expired?
- Free tunnels expire after 2 hours
- Set `NGROK_AUTH_TOKEN` for longer sessions
- Or just re-run Step 6

### Processing fails?
- Check your internet connection
- Verify API key is correct
- Try rule-based mode first

---

## 📁 Output Files

All results are saved in the `outputs/` directory:

In [ ]:
# List generated output files
print("📁 Generated output files:")
!find outputs/ -type f -name "*.json" -o -name "*.html" 2>/dev/null | sort

In [ ]:
# Download output files to your computer
from google.colab import files
import glob

print("📥 Available files for download:")

# Find all output files
output_files = glob.glob('outputs/**/*.json', recursive=True) + glob.glob('outputs/**/*.html', recursive=True)

if output_files:
    for i, file_path in enumerate(output_files[-5:], 1):  # Show last 5 files
        print(f"  {i}. {file_path}")
    
    # Uncomment to download the latest JSON file:
    # if output_files:
    #     latest_file = max(output_files, key=os.path.getctime)
    #     files.download(latest_file)
    
    print("\n💡 Uncomment the lines above to download files")
else:
    print("  No output files found. Process some content first!")

---

## 🌟 What's Next?

### Customize the System
- **Add new domains**: Update `data/cs_concepts.json` with your field
- **New languages**: Add lexicons in `data/` folder  
- **Tune parameters**: Modify `config.yaml` settings

### Integrate with Your Workflow
- **Batch processing**: Process entire course playlists
- **API usage**: Call the pipeline programmatically
- **Custom visualizations**: Export JSON and create your own graphs

### Contribute
- **Report issues**: https://github.com/navadeepkiran/Pedagogical_flow_extractor/issues
- **Submit improvements**: Create pull requests
- **Share results**: Tag us in your educational graph discoveries!

---

**📊 Ready to extract knowledge from educational content? Click the ngrok link above and start exploring! 🎓**